In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from datasets import Image as HFImage
from lmformatenforcer import JsonSchemaParser, TokenEnforcer, TokenEnforcerTokenizerData
from unsloth import FastVisionModel


/storage/dremov/diploma/conda_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
BASE_PATH = Path(".")

IMAGES_DIR = Path("/storage/dremov/diploma/datasets/test_vlm_300")
LABELS_XLSX = BASE_PATH / "5_1_features_for_vlm.xlsx"
STATS_XLSX = BASE_PATH / "5_1_statistics_analysis.xlsx"

MODEL_NAME = "unsloth/gemma-4-E4B-it"
MODEL_NAME_SLUG = MODEL_NAME.split("/")[-1].lower().replace("-", "_")

HF_CACHE_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

MAX_SEQ_LENGTH = 4096
# LORA_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_lora"
LORA_DIR = LORA_DIR = BASE_PATH / "gemma4_vlm_lora"

PREDICTIONS_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
OUT_BASE_JSON = PREDICTIONS_DIR / f"{MODEL_NAME_SLUG}_predictions_base.json"
OUT_TUNED_JSON = PREDICTIONS_DIR / f"{MODEL_NAME_SLUG}_predictions_tuned.json"


In [3]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def index_images(images_dir):
    index = {}
    if not images_dir.is_dir():
        return index
    for p in images_dir.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        index.setdefault(p.stem, p)
    return index


def read_labels(xlsx_path):
    df = pd.read_excel(xlsx_path, dtype=str).fillna("")
    id_col = df.columns[0]
    df[id_col] = df[id_col].astype(str).str.strip()
    df = df[df[id_col] != ""].copy()
    df = df.rename(columns={id_col: "identifier"})
    return df


In [ ]:
from typing import Any, Dict, List

STATS_XLSX_PATH = BASE_PATH / "5_1_statistics_analysis.xlsx"
RESPONSE_FORMAT_JSON_PATH = BASE_PATH / "response_format.json"

use_my_own_rf = True
MY_OWN_RESPONSE_FORMAT_JSON = """
{
  "schema": {
    "type": "object",
    "additionalProperties": false,
    "required": [
      "Размер рисунка",
      "Способ изображения",
      "Аккуратность рисунка",
      "Штриховка",
      "Детализированность",
      "Стирания, исправления",
      "Устойчивость изображенного человека",
      "Ресницы",
      "Зубы",
      "Дополнительные детали на лице (брови)",
      "Дополнительные детали на лице (усы и / или борода)",
      "Положение рук (подняты)",
      "Положение рук (за спиной или в карманах)",
      "Пол (признаки пола подчеркнуты)",
      "Одежда (реалистичное изображение)"
    ],
    "properties": {
      "Размер рисунка": {
        "type": "string",
        "enum": [
          "крупный",
          "стандарт",
          "мелкий"
        ],
        "description": "Определи общий размер рисунка."
      },
      "Способ изображения": {
        "type": "string",
        "enum": [
          "дифференцированная схема",
          "пластический",
          "промежуточный",
          "пластический с элементами схемы",
          "примитивная схема",
          "схема с элементами пластического",
          "условное изображение"
        ],
        "description": "Определи способ изображения персонажа на рисунке."
      },
      "Аккуратность рисунка": {
        "type": "string",
        "enum": [
          "средняя (стандарт)",
          "низкая",
          "высокая"
        ],
        "description": "Определи аккуратность, с которой выполнен рисунок."
      },
      "Штриховка": {
        "type": "string",
        "enum": [
          "нет",
          "есть"
        ],
        "description": "Определи, имеются ли заштрихованные области на рисунке."
      },
      "Детализированность": {
        "type": "string",
        "enum": [
          "средняя (стандарт)",
          "низкая",
          "высокая"
        ],
        "description": "Определи общий уровень детализированности рисунка."
      },
      "Стирания, исправления": {
        "type": "string",
        "enum": [
          "отсутствуют",
          "слабо выражены",
          "средняя выраженность",
          "сильно выражены"
        ],
        "description": "Определи, имеются ли стирания или исправления на рисунке."
      },
      "Устойчивость изображенного человека": {
        "type": "string",
        "enum": [
          "вполне устойчив",
          "относительно устойчив",
          "'висит в воздухе'",
          "неустойчив"
        ],
        "description": "Определи устойчивость изображенного человека (персонажа)."
      },
      "Ресницы": {
        "type": "string",
        "enum": [
          "отсутствуют",
          "имеются, без особенностей",
          "",
          "длинные, загнутые",
          "нарисованы небрежно",
          "другое"
        ],
        "description": "Определи, имеются ли ресницы на лице изображённого персонажа."
      },
      "Зубы": {
        "type": "string",
        "enum": [
          "отсутствуют",
          "",
          "нарисованы, без особенностей",
          "в большом количестве",
          "острые",
          "другое",
          "выделены нажимом"
        ],
        "description": "Определи, имеются ли зубы в области рта у изображённого персонажа."
      },
      "Дополнительные детали на лице (брови)": {
        "type": "string",
        "enum": [
          "",
          "брови"
        ],
        "description": "Определи, имеются ли брови в области глаз у изображённого персонажа."
      },
      "Дополнительные детали на лице (усы и / или борода)": {
        "type": "string",
        "enum": [
          "",
          "усы и / или борода"
        ],
        "description": "Определи, имеются ли усы и / или борода в области рта или подбородка у изображённого персонажа."
      },
      "Положение рук (подняты)": {
        "type": "string",
        "enum": [
          "",
          "подняты"
        ],
        "description": "Определи положение рук у изображённого персонажа. Подняты ли они?"
      },
      "Положение рук (за спиной или в карманах)": {
        "type": "string",
        "enum": [
          "",
          "за спиной или в карманах"
        ],
        "description": "Определи положение рук изображённого персонажа. Руки находятся за спиной или в карманах?"
      },
      "Пол (признаки пола подчеркнуты)": {
        "type": "string",
        "enum": [
          "",
          "признаки пола подчеркнуты"
        ],
        "description": "Определи, подчёркнуты ли признаки пола (половые органы, грудь, бёдра) у изображённого персонажа."
      },
      "Одежда (реалистичное изображение)": {
        "type": "string",
        "enum": [
          "",
          "реалистичное изображение"
        ],
        "description": "Определи, реалистична ли одежда у изображённого персонажа, если она вообще имеется?"
      }
    }
  }
}
"""


BASE_INSTRUCTION = (
    "Задача: проанализировать детский рисунок. "
    "Верни результат строго в JSON формате."
)


def build_response_format(stats_path: Path) -> Dict[str, Any]:
    df = pd.read_excel(stats_path, header=0, dtype=str).fillna("")
    value_cols = [c for c in df.columns if str(c).startswith("Значение_")]

    properties: Dict[str, Any] = {}
    required: List[str] = []

    for _, row in df.iterrows():
        feature = str(row.iloc[0]).strip()
        if not feature:
            continue
        values: List[str] = []
        seen = set()
        for c in value_cols:
            v = str(row[c]).strip().strip("'\"")
            if v and v not in seen:
                values.append(v)
                seen.add(v)
        values = ["" if x == "пусто" else x for x in values]
        values = list(dict.fromkeys(values))
        if not values:
            continue
        properties[feature] = {
            "type": "string",
            "enum": values,
            "description": f"Значение признака '{feature}'.",
        }
        required.append(feature)

    return {
        "instruction": BASE_INSTRUCTION,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": required,
            "properties": properties,
        },
    }


if use_my_own_rf:
    response_format = json.loads(MY_OWN_RESPONSE_FORMAT_JSON)
    response_format.setdefault("instruction", BASE_INSTRUCTION)
else:
    response_format = build_response_format(STATS_XLSX_PATH)
RESPONSE_FORMAT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
RESPONSE_FORMAT_JSON_PATH.write_text(
    json.dumps(response_format, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

instruction = (
    response_format["instruction"]
    + "\n\n"
    + json.dumps(response_format["schema"], ensure_ascii=False, indent=2)
)
REQUIRED_KEYS = list(response_format["schema"]["required"])

FREE_PROMPT_FROM_TRAIN = (
    "Ты - профессиональный детский психолог. Объясни, что видишь на рисунке. "
    "Проведи подробный психологический анализ."
)
STRUCTURED_PROMPT = instruction
ACTIVE_PROMPT = STRUCTURED_PROMPT


In [5]:
REQUIRED_KEYS


['Размер рисунка',
 'Способ изображения',
 'Аккуратность рисунка',
 'Штриховка',
 'Детализированность',
 'Стирания, исправления',
 'Устойчивость изображенного человека',
 'Ресницы',
 'Зубы',
 'Дополнительные детали на лице (брови)',
 'Дополнительные детали на лице (усы и / или борода)',
 'Положение рук (подняты)',
 'Положение рук (за спиной или в карманах)',
 'Пол (признаки пола подчеркнуты)',
 'Одежда (реалистичное изображение)']

In [6]:
img_index = index_images(IMAGES_DIR)
labels_df = read_labels(LABELS_XLSX)

matched_df = labels_df.copy()
matched_df["image_path"] = matched_df["identifier"].map(lambda x: img_index.get(str(x)))
matched_df = matched_df[matched_df["image_path"].notna()].copy()
matched_df["image_path"] = matched_df["image_path"].map(lambda p: str(Path(p).resolve()))

matched_df.head(10)


,identifier,Размер рисунка,Способ изображения,Аккуратность рисунка,Штриховка,Детализированность,"Стирания, исправления",Устойчивость изображенного человека,Ресницы,Зубы,Дополнительные детали на лице (брови),Дополнительные детали на лице (усы и / или борода),Положение рук (подняты),Положение рук (за спиной или в карманах),Пол (признаки пола подчеркнуты),Одежда (реалистичное изображение),image_path
56,ЧУВ5лмдс_13,мелкий,примитивная схема,низкая,нет,низкая,отсутствуют,неустойчив,,,,,,,,,/storage/dremov/diploma/datasets/test_vlm_300/...
194,ЧУВ11лж5кл_154,стандарт,пластический,низкая,есть,высокая,средняя выраженность,вполне устойчив,,отсутствуют,,,,,,реалистичное изображение,/storage/dremov/diploma/datasets/test_vlm_300/...
231,МО_6лждс_8,мелкий,дифференцированная схема,средняя (стандарт),нет,средняя (стандарт),отсутствуют,вполне устойчив,"длинные, загнутые",отсутствуют,,,,,,,/storage/dremov/diploma/datasets/test_vlm_300/...
246,МО_5лждс_168,стандарт,условное изображение,средняя (стандарт),нет,низкая,сильно выражены,'висит в воздухе',отсутствуют,отсутствуют,,,,,,,/storage/dremov/diploma/datasets/test_vlm_300/...
357,ЧУВ10лм4кл_2,мелкий,пластический,средняя (стандарт),есть,высокая,средняя выраженность,вполне устойчив,отсутствуют,отсутствуют,,,,,признаки пола подчеркнуты,реалистичное изображение,/storage/dremov/diploma/datasets/test_vlm_300/...
363,ЧУВ5лмдс_16,крупный,дифференцированная схема,средняя (стандарт),нет,средняя (стандарт),отсутствуют,неустойчив,отсутствуют,отсутствуют,,,,,,,/storage/dremov/diploma/datasets/test_vlm_300/...
370,ЧУВ9лм3кл_42,стандарт,пластический,средняя (стандарт),есть,высокая,слабо выражены,вполне устойчив,отсутствуют,отсутствуют,брови,,,,,реалистичное изображение,/storage/dremov/diploma/datasets/test_vlm_300/...
379,ЧУВ9лм3кл_35,мелкий,схема с элементами пластического,низкая,нет,низкая,средняя выраженность,вполне устойчив,отсутствуют,,,,,,,,/storage/dremov/diploma/datasets/test_vlm_300/...
438,ЧУВ9лж3кл_61,крупный,схема с элементами пластического,средняя (стандарт),нет,средняя (стандарт),слабо выражены,относительно устойчив,"имеются, без особенностей",отсутствуют,,,,,признаки пола подчеркнуты,,/storage/dremov/diploma/datasets/test_vlm_300/...
466,САМ_5_12лж6кл_1,стандарт,промежуточный,высокая,нет,средняя (стандарт),отсутствуют,'висит в воздухе',отсутствуют,отсутствуют,,,,,,реалистичное изображение,/storage/dremov/diploma/datasets/test_vlm_300/...


In [7]:
dataset = Dataset.from_pandas(matched_df, preserve_index=False)
dataset = dataset.cast_column("image_path", HFImage(mode="RGB"))
dataset = dataset.rename_column("image_path", "image")


In [8]:
def extract_json_object(text):
    if not isinstance(text, str) or not text.strip():
        return None
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start : end + 1].strip()
    return candidate if candidate.startswith("{") and candidate.endswith("}") else None


def parse_model_json(raw_text):
    obj = extract_json_object(raw_text)
    if obj is None:
        return None, "no_json_object"
    try:
        parsed = json.loads(obj)
        if not isinstance(parsed, dict):
            return None, "json_not_object"
        return parsed, None
    except Exception as e:
        return None, f"json_loads_error: {type(e).__name__}"


In [ ]:
ENFORCE_JSON_SCHEMA = True
VALIDATE_WITH_PYDANTIC = False


def get_text_tokenizer(tokenizer_or_processor):
    if hasattr(tokenizer_or_processor, "encode") and hasattr(tokenizer_or_processor, "decode"):
        return tokenizer_or_processor
    for attr in ("tokenizer", "text_tokenizer"):
        t = getattr(tokenizer_or_processor, attr, None)
        if t is not None and hasattr(t, "encode") and hasattr(t, "decode"):
            return t


def build_prefix_allowed_tokens_fn(tokenizer, schema):
    text_tokenizer = get_text_tokenizer(tokenizer)

    parser = JsonSchemaParser(schema)

    token_0 = text_tokenizer.encode("0")[-1]
    vocab_size = len(text_tokenizer)
    special_ids = set(getattr(text_tokenizer, "all_special_ids", []) or [])

    regular_tokens = []
    for token_idx in range(vocab_size):
        if token_idx in special_ids:
            continue
        decoded_after_0 = text_tokenizer.decode([token_0, token_idx])[1:]
        decoded_regular = text_tokenizer.decode([token_idx])
        is_word_start_token = len(decoded_after_0) > len(decoded_regular)
        regular_tokens.append((token_idx, decoded_after_0, is_word_start_token))

    def decode_fn(tokens):
        decoded = text_tokenizer.decode(tokens)
        return decoded.rstrip("�")

    tokenizer_data = TokenEnforcerTokenizerData(
        regular_tokens,
        decode_fn,
        getattr(text_tokenizer, "eos_token_id", None),
        False,
        vocab_size,
    )

    token_enforcer = TokenEnforcer(tokenizer_data, parser)

    def prefix_allowed_tokens_fn(batch_id, sent):
        token_sequence = sent.tolist()
        return token_enforcer.get_allowed_tokens(token_sequence).allowed_tokens

    return prefix_allowed_tokens_fn


def validate_pred_against_schema(pred, schema):
    if not isinstance(pred, dict):
        return "pred_not_object"

    props = schema.get("properties") or {}
    required = list(schema.get("required") or [])

    extra = sorted(set(pred.keys()) - set(required))
    if extra:
        return "extra_keys"

    missing = [k for k in required if k not in pred]
    if missing:
        return "missing_keys"

    for k in required:
        allowed = (props.get(k) or {}).get("enum") or []
        v = pred.get(k)
        if v not in allowed:
            return "value_not_in_enum"

    return None


In [10]:
def count_label_hits(pred, gold, keys):
    if not isinstance(gold, dict):
        return 0
    if not isinstance(pred, dict):
        return 0
    hits = 0
    for k in keys:
        if str(pred.get(k, "")).strip() == str(gold.get(k, "")).strip():
            hits += 1
    return hits
def print_inference_result(tag, idx, total, identifier, parsed, gold, raw_text, err, keys):
    print(f"| {tag} |")
    print(f" idx={idx}/{total}  identifier={identifier!r}")
    if err:
        print(f" parse_error={err}")
    vlm_payload = parsed if isinstance(parsed, dict) else raw_text
    print("VLM: предсказание модели в виде json")
    if isinstance(vlm_payload, dict):
        print(json.dumps(vlm_payload, ensure_ascii=False))
    else:
        print(vlm_payload)
    print("True: истинные метки из xlsx")
    if isinstance(gold, dict):
        print(json.dumps(gold, ensure_ascii=False))
    else:
        print("{}")
    hits = count_label_hits(parsed, gold, keys)
    print(f"Кол-во попаданий: {hits}/{len(keys)}")
    print(f"| {tag} |")
def run_inference_and_save_predictions(
    model,
    tokenizer,
    ds,
    out_json,
    tag,
    *,
    save_gold=True,
    save_raw=True,
    show_raw=True,
    show_every=1,
    enforce_json_schema=False,
    validate_with_pydantic=False,
    prompt_text=ACTIVE_PROMPT,
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    FastVisionModel.for_inference(model)
    schema = response_format["schema"]
    prefix_allowed_tokens_fn = (
        build_prefix_allowed_tokens_fn(tokenizer, schema) if enforce_json_schema else None
    )
    rows = []
    for i in range(len(ds)):
        ex = ds[i]
        image = ex["image"]
        identifier = str(ex.get("identifier", i))
        image_path = ex.get("image_path") or ex.get("image")
        gold = {k: str(ex.get(k, "")) for k in REQUIRED_KEYS} if save_gold else None
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt_text},
                ],
            }
        ]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs = tokenizer(
            image,
            input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to(device)
        gen_kwargs = {
            **inputs,
            "max_new_tokens": 512,
            "use_cache": True,
        }
        if prefix_allowed_tokens_fn is not None:
            gen_kwargs.update(
                {
                    "do_sample": False,
                    "prefix_allowed_tokens_fn": prefix_allowed_tokens_fn,
                }
            )
        else:
            gen_kwargs.update(
                {
                    "temperature": 0.2,
                    "top_p": 0.9,
                    "top_k": 64,
                }
            )
        with torch.inference_mode():
            gen_ids = model.generate(**gen_kwargs)
        new_tokens = gen_ids[0][inputs["input_ids"].shape[-1] :]
        raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        parsed, err = parse_model_json(raw_text)
        if parsed is not None and validate_with_pydantic:
            verr = validate_pred_against_schema(parsed, schema)
            if verr is not None:
                err = f"schema_validation_error: {verr}"
        if show_raw and show_every > 0 and (i % show_every == 0):
            print_inference_result(
                tag=tag,
                idx=i + 1,
                total=len(ds),
                identifier=identifier,
                parsed=parsed,
                gold=gold,
                raw_text=raw_text,
                err=err,
                keys=REQUIRED_KEYS,
            )
        row = {
            "tag": tag,
            "idx": i,
            "identifier": identifier,
            "parse_error": err,
            "pred": parsed,
        }
        if image_path is not None:
            row["image"] = str(image_path)
        if save_gold:
            row["gold"] = gold
        if save_raw:
            row["raw"] = raw_text
        rows.append(row)
    out_json.parent.mkdir(parents=True, exist_ok=True)
    out_json.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    return rows


## 1) Инференс на базовой модели (Gemma4)


In [ ]:
base_model, base_tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    trust_remote_code=True,
)

base_predictions = run_inference_and_save_predictions(
    base_model,
    base_tokenizer,
    dataset,
    OUT_BASE_JSON,
    tag="base",
    save_gold=True,
    save_raw=True,
    show_raw=True,
    show_every=1,
    enforce_json_schema=ENFORCE_JSON_SCHEMA,
    validate_with_pydantic=VALIDATE_WITH_PYDANTIC,
)

pd.DataFrame(
    [
        {
            "idx": r["idx"],
            "identifier": r["identifier"],
            "parse_error": r["parse_error"],
        }
        for r in base_predictions
    ]
).head(10)


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.4.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla V100-PCIE-32GB. Num GPUs = 1. Max memory: 31.733 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|████████████████████████████████████████████████████████| 2130/2130 [00:03<00:00, 594.03it/s]


| base |
 idx=1/300  identifier='ЧУВ5лмдс_13'
VLM: предсказание модели в виде json
{"Размер рисунка": "мелкий", "Способ изображения": "примитивная схема", "Аккуратность рисунка": "низкая", "Штриховка": "нет", "Детализированность": "низкая", "Стирания, исправления": "отсутствуют", "Устойчивость изображенного человека": "относительно устойчив", "Ресницы": "отсутствуют", "Зубы": "отсутствуют", "Дополнительные детали на лице (брови)": "", "Дополнительные детали на лице (усы и / или борода)": "", "Положение рук (подняты)": "", "Положение рук (за спиной или в карманах)": "", "Пол (признаки пола подчеркнуты)": "", "Одежда (реалистичное изображение)": ""}
True: истинные метки из xlsx
{"Размер рисунка": "мелкий", "Способ изображения": "примитивная схема", "Аккуратность рисунка": "низкая", "Штриховка": "нет", "Детализированность": "низкая", "Стирания, исправления": "отсутствуют", "Устойчивость изображенного человека": "неустойчив", "Ресницы": "", "Зубы": "", "Дополнительные детали на лице (брови

## 2) Инференс на дообученной модели (LoRA)


In [ ]:
tuned_model, tuned_tokenizer = FastVisionModel.from_pretrained(
    model_name=str(LORA_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    trust_remote_code=True,
)

tuned_predictions = run_inference_and_save_predictions(
    tuned_model,
    tuned_tokenizer,
    dataset,
    OUT_TUNED_JSON,
    tag="tuned",
    save_gold=True,
    save_raw=True,
    show_raw=True,
    show_every=1,
    enforce_json_schema=ENFORCE_JSON_SCHEMA,
    validate_with_pydantic=VALIDATE_WITH_PYDANTIC,
)

pd.DataFrame(
    [
        {
            "idx": r["idx"],
            "identifier": r["identifier"],
            "parse_error": r["parse_error"],
        }
        for r in tuned_predictions
    ]
).head(10)


## 3) Графики по сохранённым метрикам


In [ ]:
import matplotlib.pyplot as plt

METRICS_DIR = BASE_PATH / f"{MODEL_NAME_SLUG}_metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_METRICS_CSV = METRICS_DIR / f"{MODEL_NAME_SLUG}_metrics_combined.csv"
PLOTS_DIR = METRICS_DIR / f"{MODEL_NAME_SLUG}_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if COMBINED_METRICS_CSV.exists():
    df = pd.read_csv(COMBINED_METRICS_CSV)

    required_cols = {"tag", "key", "accuracy", "f1_micro", "f1_macro", "f1_weighted"}
    missing = sorted(required_cols - set(df.columns))
    if missing:
        pd.DataFrame({"error": ["metrics_combined.csv missing columns"], "missing": [missing]})
    else:
        per_key = df[df["key"] != "__overall__"].copy()
        overall = df[df["key"] == "__overall__"].copy()

        def plot_metric_bars(metric):
            pivot = per_key.pivot_table(index="key", columns="tag", values=metric, aggfunc="mean")
            pivot = pivot.sort_values(by=pivot.columns[0], ascending=False)

            ax = pivot.plot(kind="bar", figsize=(14, 6))
            ax.set_title(f"Gemma4: сравнение по признакам: {metric} (base vs tuned)")
            ax.set_xlabel("Признак")
            ax.set_ylabel(metric)
            ax.set_ylim(0.0, 1.0)
            ax.grid(True, axis="y", linestyle="--", alpha=0.35)
            plt.xticks(rotation=45, ha="right")
            plt.tight_layout()
            out = PLOTS_DIR / f"per_key_{metric}.png"
            plt.savefig(out, dpi=150)
            plt.close()

        for m in ("accuracy", "f1_micro", "f1_macro", "f1_weighted"):
            plot_metric_bars(m)

        if not overall.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            o = overall.pivot_table(index="tag", values=["accuracy", "f1_micro", "f1_macro", "f1_weighted"], aggfunc="mean")
            o.plot(kind="bar", ax=ax, rot=0)
            ax.set_title("Gemma4: общие метрики (base vs tuned)")
            ax.set_ylim(0.0, 1.0)
            ax.grid(True, axis="y", linestyle="--", alpha=0.35)
            plt.tight_layout()
            plt.savefig(PLOTS_DIR / "overall.png", dpi=150)
            plt.close()

        overall
else:
    pass


In [ ]:
OUT_BASE_JSON, OUT_TUNED_JSON, OUT_BASE_JSON.exists(), OUT_TUNED_JSON.exists()
